0. Imports.

In [2]:
import pandas as pd
from sqlalchemy import create_engine

1. Carregando os dados.

In [3]:
tables = {
    'customers': 'olist_order_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

olist_df = {}
for name, archive in tables.items():
    df = pd.read_csv(f'../data/raw/{archive}')
    olist_df[name] = df

2. olist_orders_dataset.csv.

- 2.1. Convertendo tipos de dados:
  *  Convertendo as colunas a partir de order_purchase_timestamp de str e para datetime.

In [4]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    olist_df['orders'][col] = pd.to_datetime(olist_df['orders'][col])


- 2.2. Criando colunas:
  *  delivery_delay_days: dias de atraso (int) na entrega, essencial para as perguntas de negócio 1, 2, 3, 12 e 14;
  *  order_processing_time_days: dias entre compra e aprovação (int), útil para a pergunta de negócio 5;
  *  purchase_year_month: ano e mês da compra (str), facilita agregações temporais no SQL para as perguntas 1 e 6.

In [5]:
#Criando as colunas
olist_df['orders']['delivery_delay_days'] = (olist_df['orders']['order_delivered_customer_date'] - olist_df['orders']['order_estimated_delivery_date']).dt.days
olist_df['orders']['order_processing_time_days'] = (olist_df['orders']['order_approved_at'] - olist_df['orders']['order_purchase_timestamp']).dt.days
olist_df['orders']['purchase_year_month'] = olist_df['orders']['order_purchase_timestamp'].dt.to_period('M').astype(str)

- 2.3. Valores nulos: Os nulos nas colunas 'order_approved_at', 'order_delivered_carrier_date' e 'order_delivered_customer_date' têm significado lógico (pedidos cancelados, não aprovados ou não entregues), portanto, os valores devem ser mantidos;
- 2.4. Nomes de colunas: Já estão padronizados;
- 2.5. Colunas duplicadas: Não há.

3. olist_products_dataset.

- 3.1. Valores nulos:
  *  product_category_name: remover linhas nulas, pois é foco das perguntas de negócio 3, 7 e 13;
  *  product_name_lenght, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm: imputar pela mediana, pois os nulos são numéricos sem significado lógico claro.

In [6]:
#Removendo linhas nulas em product_category_name (foco das perguntas de negócio)
olist_df['products'] = olist_df['products'].dropna(subset=['product_category_name'])

#Imputando pela mediana nas colunas numéricas restantes
numeric_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in numeric_cols:
    median = olist_df['products'][col].median()
    olist_df['products'][col] = olist_df['products'][col].fillna(median)

- 3.2. Tipos de dados: nada a mudar, todas as colunas estão com tipos corretos;
- 3.3. Criar colunas: Sem necessidade;
- 3.4. Nomes de colunas: Já estão padronizados;
- 3.5 Colunas duplicadas: Não há.

4. olist_sellers_dataset.

- 4.1. Valores nulos: Não há;
- 4.2. Tipos de dados: Nada a mudar, todas as colunas estão com tipos corretos;
- 4.3. Criar colunas: Sem necessidade;
- 4.4. Nomes de colunas: Já estão padronizados;
- 4.5. Colunas duplicadas: Não há.

Obs.: seller_city possui inconsistências (como: 'lages - sc' e 'rio de janeiro, rio de janeiro, brasil'), portanto é necessário remover caracteres especiais, espaços extras e padronizar tudo em minúsculo para garantir consistência.

In [7]:
olist_df['sellers']['seller_city'] = (
    olist_df['sellers']['seller_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

5. product_category_name_translation.

- 5.1. Valores nulos: Não há;
- 5.2. Tipos de dados: Nada a mudar, todas as colunas estão com tipos corretos;
- 5.3. Criar colunas: Sem necessidade;
- 5.4. Nomes de colunas: Já estão padronizados;
- 5.5. Colunas duplicadas: Não há.

6. olist_order_items_dataset.

- 6.1. Tipos de dados: 
  *   shipping_limit_date: Convertendo de str para datetime.


In [9]:
olist_df['order_items']['shipping_limit_date'] = pd.to_datetime(olist_df['order_items']['shipping_limit_date'])

- 6.2. Criar colunas: 
  *   total_item_value: Valor total pago por item incluindo frete, útil para as perguntas de negócios 4 e 7.

In [10]:
olist_df['order_items']['total_item_value'] = olist_df['order_items']['price'] + olist_df['order_items']['freight_value']

- 6.3. Valores nulos: Não há;
- 6.4. Nomes de colunas: Já estão padronizados;
- 6.5. Colunas duplicadas: Não há.

7. olist_order_payments_dataset.

- 7.1. Valores nulos: não há;
- 7.2. Tipos de dados: Nada a mudar, todas as colunas estão com tipos corretos;
- 7.3. Criar colunas: Sem necessidade;
- 7.4. Nomes de colunas: Já estão padronizados;
- 7.5. Colunas duplicadas: Não há.

8. olist_order_reviews_dataset.

- 8.1. Valores nulos:
  *  review_comment_title: Nenhuma das 16 perguntas de negócio foca no conteúdo textual dos títulos. Portanto, preencher com 'Sem título';
  *  review_comment_message: Nenhuma das 16 perguntas de negócio foca no conteúdo textual das mensagens. Portanto, preencher com 'Sem mensagem'.

In [11]:
#Imputandos as linhas com valores nulos
olist_df['order_reviews']['review_comment_title'] = olist_df['order_reviews']['review_comment_title'].fillna('Sem título')
olist_df['order_reviews']['review_comment_message'] = olist_df['order_reviews']['review_comment_message'].fillna('Sem mensagem')

- 8.2. Tipos de dados:
  *  review_creation_date: Convertendo de str para datetime;
  *  review_answer_timestamp: Convertendo de str para datetime.

In [12]:
#Convertendo tipos de colunas
olist_df['order_reviews']['review_creation_date'] = pd.to_datetime(olist_df['order_reviews']['review_creation_date'])
olist_df['order_reviews']['review_answer_timestamp'] = pd.to_datetime(olist_df['order_reviews']['review_answer_timestamp'])

- 8.3. Criar colunas: Sem necessidade;
- 8.4. Nomes de colunas: Já estão padronizados;
- 8.5. Colunas duplicadas: Não há.

9. olist_order_customers_dataset

- 9.1. Valores nulos: não há;
- 9.2. Tipos de dados: Nada a mudar, todas as colunas estão com tipos corretos;
- 9.3. Criar colunas: Sem necessidade;
- 9.4. Nomes de colunas: Já estão padronizados;
- 9.5. Colunas duplicadas: Não há.

Obs.: customer_city possui inconsistências, portanto é necessário remover caracteres especiais, espaços extras e padronizar tudo em minúsculo para garantir consistência.

In [13]:
olist_df['customers']['customer_city'] = (
    olist_df['customers']['customer_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

10. Exportando os arquivos csv.

In [14]:
for name, df in olist_df.items():
    df.to_csv(f'../data/processed/{name}.csv', index=False)

11. Conectando ao MySQL.

In [15]:
#Configurações do MySQL
username = "root"
password = "" 
host = "localhost"
port = "3306"
database = "olist_brazilian_ecommerce" #Banco criado previamente

#Conexão
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

#Envio para o Banco
for name, df in olist_df.items():
    df.to_sql(name, engine, if_exists="replace", index=False)